# 7b. Single-Cell QC

## Purpose
Flag low-quality single cells per patient using three criteria applied in cascade:
1. **NaN detection** — cells missing key metadata or feature values
2. **Inherited organoid flags** — cells whose parent organoid was flagged in `7a`
3. **Nucleus outliers** — abnormally small/large nuclei or high mass displacement

Outlier detection (step 3) only runs on cells that passed steps 1 and 2.

This is **step 7b of Stage 4 (image-based profiling)**. It runs once per patient
and depends on `7a.organoid_qc.ipynb` having run first.

## Inputs
- `data/{patient}/image_based_profiles/3.annotated_profiles/sc_anno.parquet`
- `data/{patient}/image_based_profiles/4.qc_profiles/organoid_flagged_outliers.parquet`

## Outputs
- `data/{patient}/image_based_profiles/4.qc_profiles/sc_flagged_outliers.parquet`
  — SC profile with added `Metadata_cqc_*` flag columns

## Notes
- QC flags are additive: a cell can be flagged by multiple criteria simultaneously.
- The `Metadata_cqc_organoid_flagged` column propagates organoid-level flags down
  to all cells belonging to that organoid, linking 7a and 7b outputs.

In [1]:
import os
import pathlib

import pandas as pd
from cosmicqc import find_outliers
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
profile_base_dir = root_dir

In [2]:
if not in_notebook:
    args = parse_args()
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0014_T1"
    image_based_profiles_subparent_name = "image_based_profiles"

## Load profiles and initialize QC flags

QC is applied in three rounds:
1. **NaN detection** (`Metadata_cqc_nan_detected`) — missing ObjectID, volume, or parent
2. **Inherited organoid flags** (`Metadata_cqc_organoid_flagged`, `Metadata_cqc_missing_parent_organoid`)
   — cells whose parent organoid failed QC in 7a, or have no parent organoid at all
3. **Nucleus outliers** — applied only to cells that passed rounds 1 and 2

In [3]:
sc_file = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "3.annotated_profiles/sc_anno.parquet"
)
organoid_file = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "4.qc_profiles/organoid_flagged_outliers.parquet"
)

nucleocentric_annotated_sammed_path = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "3.annotated_profiles/nucleocentric_sammed_anno.parquet"
).resolve()
nucleocentric_annotated_morphem_output_path = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "3.annotated_profiles/nucleocentric_morphem_anno.parquet"
).resolve()
sammed_annotated_sc_profiles_path = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "3.annotated_profiles/sammed_sc_anno.parquet"
).resolve()


output_dir = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "4.qc_profiles"
)
output_dir.mkdir(parents=True, exist_ok=True)

sc_qc_output_path = pathlib.Path(f"{output_dir}/sc_flagged_outliers.parquet").resolve()
sammed_sc_qc_output_path = pathlib.Path(
    f"{output_dir}/sammed_sc_flagged_outliers.parquet"
).resolve()
nucleocentric_sammed_qc_output_path = pathlib.Path(
    f"{output_dir}/nucleocentric_sammed_flagged_outliers.parquet"
).resolve()
nucleocentric_morphem_qc_output_path = pathlib.Path(
    f"{output_dir}/nucleocentric_morphem_flagged_outliers.parquet"
).resolve()

orig_sc_profiles_df = pd.read_parquet(sc_file)
organoid_qc_profiles_df = pd.read_parquet(organoid_file)
# Print the shape and head of the combined organoid profiles DataFrame
print(orig_sc_profiles_df.shape)
orig_sc_profiles_df

(2596, 2650)


,Metadata_Biology_PatientTumor,Metadata_Experiment_Class,Metadata_Experiment_Dose,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,Metadata_Experiment_Unit,Metadata_Experiment_Well,Metadata_Experiment_WellFOV,Metadata_Location_Cell_CenterX,...,Nuclei_Mito_Texture_Variance-3-09-256,Nuclei_Mito_Texture_Variance-3-10-256,Nuclei_Mito_Texture_Variance-3-11-256,Nuclei_Mito_Texture_Variance-3-12-256,Nuclei_NoChannel_AreaSizeShape_BboxVolume,Nuclei_NoChannel_AreaSizeShape_EquivalentDiameter,Nuclei_NoChannel_AreaSizeShape_EulerNumber,Nuclei_NoChannel_AreaSizeShape_Extent,Nuclei_NoChannel_AreaSizeShape_SurfaceArea,Nuclei_NoChannel_AreaSizeShape_Volume
0,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,255.746097,...,0.000000,0.000000,0.000000e+00,7.748604e-304,5244.0,19.820474,1,0.777460,19.962237,4077.0
1,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,635.240003,...,0.000000,0.000000,0.000000e+00,7.748604e-304,259920.0,60.983233,1,0.456867,622.313679,118749.0
2,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,565.496171,...,0.000000,0.000000,0.000000e+00,7.748604e-304,75710.0,36.442039,1,0.334698,162.769658,25340.0
3,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,1338.730872,...,0.000000,0.000000,0.000000e+00,7.748604e-304,164268.0,58.728967,1,0.645658,449.452880,106061.0
4,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,539.489471,...,0.000000,0.000000,0.000000e+00,5.453612e-312,228105.0,62.323697,1,0.555678,655.422914,126753.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2591,NF0014_T1,Control,1,Control,Control,DMSO,%,G9,G9-2,635.748169,...,0.000000,0.000000,7.748604e-304,7.748604e-304,28179.0,31.488744,1,0.580148,42.757919,16348.0
2592,NF0014_T1,Control,1,Control,Control,DMSO,%,G9,G9-2,548.947130,...,0.000000,0.000000,7.748604e-304,7.748604e-304,239540.0,66.424230,1,0.640620,587.997186,153454.0
2593,NF0014_T1,Control,1,Control,Control,DMSO,%,G9,G9-2,851.858134,...,0.000000,0.000000,5.453612e-312,7.748604e-304,176280.0,55.665419,1,0.512333,464.552124,90314.0
2594,NF0014_T1,Control,1,Control,Control,DMSO,%,G9,G9-2,766.871117,...,0.000000,0.000000,0.000000e+00,0.000000e+00,211420.0,62.551501,1,0.606130,501.130490,128148.0


In [4]:
sc_profiles_df = orig_sc_profiles_df.copy()
sc_profiles_df["Metadata_cqc_nan_detected"] = (
    sc_profiles_df[
        [
            "Metadata_Object_ObjectID",
            "Metadata_Object_ParentOrganoid",
            "Cell_NoChannel_AreaSizeShape_Volume",
        ]
    ]
    .isna()
    .any(axis=1)
)
# Print the number of organoids flagged
flagged_count = sc_profiles_df["Metadata_cqc_nan_detected"].sum()
print(f"Number of organoids flagged: {flagged_count}")

sc_profiles_df.head()

Number of organoids flagged: 0


,Metadata_Biology_PatientTumor,Metadata_Experiment_Class,Metadata_Experiment_Dose,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,Metadata_Experiment_Unit,Metadata_Experiment_Well,Metadata_Experiment_WellFOV,Metadata_Location_Cell_CenterX,...,Nuclei_Mito_Texture_Variance-3-10-256,Nuclei_Mito_Texture_Variance-3-11-256,Nuclei_Mito_Texture_Variance-3-12-256,Nuclei_NoChannel_AreaSizeShape_BboxVolume,Nuclei_NoChannel_AreaSizeShape_EquivalentDiameter,Nuclei_NoChannel_AreaSizeShape_EulerNumber,Nuclei_NoChannel_AreaSizeShape_Extent,Nuclei_NoChannel_AreaSizeShape_SurfaceArea,Nuclei_NoChannel_AreaSizeShape_Volume,Metadata_cqc_nan_detected
0,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,255.746097,...,0.0,0.0,7.748604e-304,5244.0,19.820474,1,0.777460,19.962237,4077.0,False
1,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,635.240003,...,0.0,0.0,7.748604e-304,259920.0,60.983233,1,0.456867,622.313679,118749.0,False
2,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,565.496171,...,0.0,0.0,7.748604e-304,75710.0,36.442039,1,0.334698,162.769658,25340.0,False
3,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,1338.730872,...,0.0,0.0,7.748604e-304,164268.0,58.728967,1,0.645658,449.452880,106061.0,False
4,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,539.489471,...,0.0,0.0,5.453612e-312,228105.0,62.323697,1,0.555678,655.422914,126753.0,False


In [5]:
# Round 2: propagate organoid-level QC flags to single cells.
# A cell is flagged if its parent organoid was flagged in 7a.
# We match on (ParentOrganoid, WellFOV) rather than ParentOrganoid alone because
# object IDs are reassigned per-FOV and are not globally unique across the patient.

# Default QC flags
sc_profiles_df["Metadata_cqc_organoid_flagged"] = False
sc_profiles_df["Metadata_cqc_nan_detected"] = (
    sc_profiles_df[
        ["Metadata_Object_ObjectID", "Nuclei_NoChannel_AreaSizeShape_Volume"]
    ]
    .isna()
    .any(axis=1)
)
sc_profiles_df["Metadata_cqc_missing_parent_organoid"] = (
    sc_profiles_df["Metadata_Object_ParentOrganoid"] == -1
)


organoid_flags_df = organoid_qc_profiles_df[
    ["Metadata_Object_ObjectID", "Metadata_Experiment_WellFOV"]
    + [col for col in organoid_qc_profiles_df.columns if col.startswith("Metadata_cqc")]
]

# Get flagged (object_id, image_set) pairs
flagged_pairs = set(
    organoid_flags_df.loc[
        organoid_flags_df.filter(like="cqc").any(axis=1),
        ["Metadata_Object_ObjectID", "Metadata_Experiment_WellFOV"],
    ].itertuples(index=False, name=None)
)

# Flag SC rows where both parent_organoid & image_set match a flagged organoid
sc_profiles_df["Metadata_cqc_organoid_flagged"] = sc_profiles_df.apply(
    lambda row: (
        (row["Metadata_Object_ParentOrganoid"], row["Metadata_Experiment_WellFOV"])
        in flagged_pairs
    ),
    axis=1,
)

print(sc_profiles_df.shape)
sc_profiles_df.head()

(2596, 2653)


,Metadata_Biology_PatientTumor,Metadata_Experiment_Class,Metadata_Experiment_Dose,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,Metadata_Experiment_Unit,Metadata_Experiment_Well,Metadata_Experiment_WellFOV,Metadata_Location_Cell_CenterX,...,Nuclei_Mito_Texture_Variance-3-12-256,Nuclei_NoChannel_AreaSizeShape_BboxVolume,Nuclei_NoChannel_AreaSizeShape_EquivalentDiameter,Nuclei_NoChannel_AreaSizeShape_EulerNumber,Nuclei_NoChannel_AreaSizeShape_Extent,Nuclei_NoChannel_AreaSizeShape_SurfaceArea,Nuclei_NoChannel_AreaSizeShape_Volume,Metadata_cqc_nan_detected,Metadata_cqc_organoid_flagged,Metadata_cqc_missing_parent_organoid
0,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,255.746097,...,7.748604e-304,5244.0,19.820474,1,0.777460,19.962237,4077.0,False,False,True
1,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,635.240003,...,7.748604e-304,259920.0,60.983233,1,0.456867,622.313679,118749.0,False,False,False
2,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,565.496171,...,7.748604e-304,75710.0,36.442039,1,0.334698,162.769658,25340.0,False,False,False
3,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,1338.730872,...,7.748604e-304,164268.0,58.728967,1,0.645658,449.452880,106061.0,False,False,True
4,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,539.489471,...,5.453612e-312,228105.0,62.323697,1,0.555678,655.422914,126753.0,False,False,False


In [6]:
sc_profiles_df["Nuclei_NoChannel_AreaSizeShape_Volume"].describe()

count      2596.000000
mean      61327.827812
std       41213.033859
min         504.000000
25%       25379.250000
50%       62201.000000
75%       87489.750000
max      440030.000000
Name: Nuclei_NoChannel_AreaSizeShape_Volume, dtype: float64

## Detect outlier single-cells using the non-flagged data

We will attempt to detect instances of poor quality segmentations using the nuclei compartment as the base. The conditions we are using are as follows:

1. Abnormally small or large nuclei using `Volume`
2. Abnormally high `mass displacement` in the nuclei for instances of mis-segmentation of background/no longer in-focus

In [7]:
# Set the metadata columns to be used in the QC process
metadata_columns = [x for x in sc_profiles_df.columns if "Metadata" in x]

In [8]:
# Round 3: nucleus-based outlier detection using z-score thresholds.
# Threshold sign: negative = flag below mean, positive = flag above mean.
# Threshold magnitude: number of standard deviations from the mean.
# Only cells that passed rounds 1 and 2 are evaluated here.
# Only process the rows that are not flagged
filtered_plate_df = sc_profiles_df[
    ~(
        sc_profiles_df["Metadata_cqc_nan_detected"]
        | sc_profiles_df["Metadata_cqc_organoid_flagged"]
        | sc_profiles_df["Metadata_cqc_missing_parent_organoid"]
    )
]

# --- Find size based nuclei outliers ---
print("Finding small nuclei outliers...")
small_nuclei_outliers = find_outliers(
    df=filtered_plate_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        "Nuclei_NoChannel_AreaSizeShape_Volume": -1,  # Detect very small nuclei
    },
)

# Ensure the column exists before assignment
sc_profiles_df["Metadata_cqc_small_nuclei_outlier"] = False
sc_profiles_df.loc[small_nuclei_outliers.index, "Metadata_cqc_small_nuclei_outlier"] = (
    True
)

print("Finding large nuclei outliers...")
large_nuclei_outliers = find_outliers(
    df=filtered_plate_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        "Nuclei_NoChannel_AreaSizeShape_Volume": 2,  # Detect very large nuclei
    },
)

# Ensure the column exists before assignment
sc_profiles_df["Metadata_cqc_large_nuclei_outlier"] = False
sc_profiles_df.loc[large_nuclei_outliers.index, "Metadata_cqc_large_nuclei_outlier"] = (
    True
)

# --- Find mass displacement based nuclei outliers ---
print("Finding high mass displacement outliers...")
high_mass_displacement_outliers = find_outliers(
    df=filtered_plate_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        "Nuclei_DNA_Intensity_MassDisplacement": 2,  # Detect high mass displacement
    },
)

# Ensure the column exists before assignment
sc_profiles_df["Metadata_cqc_mass_displacement_outlier"] = False
sc_profiles_df.loc[
    high_mass_displacement_outliers.index, "Metadata_cqc_mass_displacement_outlier"
] = True

# Print number of outliers (only in filtered rows)
small_count = filtered_plate_df.index.intersection(small_nuclei_outliers.index).shape[0]
large_count = filtered_plate_df.index.intersection(large_nuclei_outliers.index).shape[0]
high_mass_count = filtered_plate_df.index.intersection(
    high_mass_displacement_outliers.index
).shape[0]

print(f"Small nuclei outliers found: {small_count}")
print(f"Large nuclei outliers found: {large_count}")
print(f"High mass displacement outliers found: {high_mass_count}")

# Save updated plate_df with flag columns included
sc_profiles_df.to_parquet(sc_qc_output_path, index=False)

Finding small nuclei outliers...
Number of outliers: 369 (21.48%)
Outliers Range:
Nuclei_NoChannel_AreaSizeShape_Volume Min: 504.0
Nuclei_NoChannel_AreaSizeShape_Volume Max: 22283.0
Finding large nuclei outliers...
Number of outliers: 44 (2.56%)
Outliers Range:
Nuclei_NoChannel_AreaSizeShape_Volume Min: 149738.0
Nuclei_NoChannel_AreaSizeShape_Volume Max: 440030.0
Finding high mass displacement outliers...
Number of outliers: 42 (2.44%)
Outliers Range:
Nuclei_DNA_Intensity_MassDisplacement Min: 5.561026
Nuclei_DNA_Intensity_MassDisplacement Max: 34.41847
Small nuclei outliers found: 369
Large nuclei outliers found: 44
High mass displacement outliers found: 42


In [9]:
sc_profiles_df.head()

,Metadata_Biology_PatientTumor,Metadata_Experiment_Class,Metadata_Experiment_Dose,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,Metadata_Experiment_Unit,Metadata_Experiment_Well,Metadata_Experiment_WellFOV,Metadata_Location_Cell_CenterX,...,Nuclei_NoChannel_AreaSizeShape_EulerNumber,Nuclei_NoChannel_AreaSizeShape_Extent,Nuclei_NoChannel_AreaSizeShape_SurfaceArea,Nuclei_NoChannel_AreaSizeShape_Volume,Metadata_cqc_nan_detected,Metadata_cqc_organoid_flagged,Metadata_cqc_missing_parent_organoid,Metadata_cqc_small_nuclei_outlier,Metadata_cqc_large_nuclei_outlier,Metadata_cqc_mass_displacement_outlier
0,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,255.746097,...,1,0.777460,19.962237,4077.0,False,False,True,False,False,False
1,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,635.240003,...,1,0.456867,622.313679,118749.0,False,False,False,False,False,False
2,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,565.496171,...,1,0.334698,162.769658,25340.0,False,False,False,False,False,False
3,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,1338.730872,...,1,0.645658,449.452880,106061.0,False,False,True,False,False,False
4,NF0014_T1,Small Molecule,1,MEK1/2 inhibitor,Kinase Inhibitor,Trametinib,uM,C10,C10-1,539.489471,...,1,0.555678,655.422914,126753.0,False,False,False,False,False,False


### Merge the qc flags to the deep learning-based profiles and save the output
Merge the QC flags back to the original single cell profiles, which will be used in downstream analyses and single cell QC. 
We need to do this beacuase we do not run qc on black-box features. 
Merge on the Metadata_Biology_PatientTumor, Metadata_Experiment_WellFOV
and the Metadata_Object_ObjectID columns, which together uniquely identify each organoid profile row.

In [10]:
nucleocentric_annotated_sammed_df = pd.read_parquet(nucleocentric_annotated_sammed_path)
nucleocentric_annotated_morphem_df = pd.read_parquet(
    nucleocentric_annotated_morphem_output_path
)
sammed_annotated_sc_profiles_df = pd.read_parquet(sammed_annotated_sc_profiles_path)
df_dict = {
    "nulceocentric_sammed": {
        "df": nucleocentric_annotated_sammed_df,
        "qc_output_path": nucleocentric_sammed_qc_output_path,
    },
    "nucleocentric_chammi": {
        "df": nucleocentric_annotated_morphem_df,
        "qc_output_path": nucleocentric_morphem_qc_output_path,
    },
    "sammed_sc_profiles": {
        "df": sammed_annotated_sc_profiles_df,
        "qc_output_path": sammed_sc_qc_output_path,
    },
}

In [11]:
# set the merge keys to int for both dataframes to ensure they match
merge_keys = [
    "Metadata_Biology_PatientTumor",
    "Metadata_Experiment_WellFOV",
    "Metadata_Object_ObjectID",
]
qc_keys = [col for col in sc_profiles_df.columns if "Metadata_cqc" in col]

for profile_name in df_dict:
    df = df_dict[profile_name]["df"]
    for key in merge_keys:
        if key not in df.columns:
            raise ValueError(f"Merge key {key} not found in dataframe columns.")
    qc_annotated_df = df.merge(
        sc_profiles_df[qc_keys + merge_keys],
        on=merge_keys,
        how="left",
    )
    if qc_annotated_df.shape[1] == df.shape[1]:
        raise ValueError(
            f"No new columns were added during the merge. Check that the merge keys {merge_keys} are correct and that the qc keys {qc_keys} are present in the sc_profiles_df."
        )
    qc_annotated_df.to_parquet(df_dict[profile_name]["qc_output_path"], index=False)